In [5]:
# ── Imports ──────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tabulate import tabulate
from mp_api.client import MPRester

# ── 1. Fetch data from Materials Project ─────────────────────────────────────

API_KEY = "4IeHY5jVcrgiKXNuAo6Jgs7yC0Z3hsli"
with MPRester(API_KEY) as mpr:
    # Query for Li-based cathode materials with high energy density
    docs = mpr.materials.insertion_electrodes.search(
    working_ion="Li", average_voltage = (0.0, 6.5), stability_charge=(0.0, 0.5), max_delta_volume=(0,1),
        # 0.1 > stability_charge (meV) starts to be unstable
    fields=[
        "battery_id", "formula_discharge", "average_voltage"
        , "energy_grav", "energy_vol", "capacity_grav", "capacity_vol", "stability_charge", "fracA_charge", "max_delta_volume", "stability_discharge", "fracA_discharge"
    ]
)


df = pd.DataFrame([doc.dict() for doc in docs])


# ── 2. Define objectives ──────────────────────────────────────────────────────

objectives = {
    "average_voltage":    "max",
    "capacity_grav":      "max",
    "energy_grav":        "max",
    "fracA_charge":       "min",
    "fracA_discharge":    "max",
    "stability_charge":   "min",
    "stability_discharge":"min",
    "max_delta_volume":   "min",
}

cols = list(objectives.keys())

# ── 3. Clean data ─────────────────────────────────────────────────────────────

df_clean = df[cols + ["battery_id", "formula_discharge"]].dropna().copy()
df_clean = df_clean.reset_index(drop=True)

print(f"Materials after cleaning: {len(df_clean)}")

# ── 4. Validate data ──────────────────────────────────────────────────────────

"""physical_bounds = {
    "average_voltage":    (0.5, 6.0),
    "capacity_grav":      (0, 1000),
    "capacity_vol":       (0, 5000),
    "energy_grav":        (0, 5000),
    "energy_vol":         (0, 15000),
    "fracA_charge":       (0, 1),
    "fracA_discharge":    (0, 1),
    "stability_charge":   (0, 1),
    "stability_discharge":(0, 1),
    "max_delta_volume":   (0, 500),
}

print("\n=== Physical bounds check ===")
for col, (lo, hi) in physical_bounds.items():
    violations = df_clean[(df_clean[col] < lo) | (df_clean[col] > hi)]
    if len(violations) > 0:
        print(f"  {col}: {len(violations)} violations — removing")
        df_clean = df_clean[~((df_clean[col] < lo) | (df_clean[col] > hi))]
    else:
        print(f"  {col}: OK")

df_clean = df_clean.reset_index(drop=True)
print(f"\nMaterials after validation: {len(df_clean)}")
"""
# ── 5. Convert to minimization ────────────────────────────────────────────────

df_min = df_clean[cols].copy()
for col, direction in objectives.items():
    if direction == "max":
        df_min[col] = -df_clean[col]

costs = df_min.values

# ── 6. Pareto front ───────────────────────────────────────────────────────────

def pareto_front_fast(costs):
    """
    Returns boolean mask of non-dominated (Pareto-optimal) solutions.
    All objectives assumed to be minimization.
    """
    costs = np.array(costs)
    n = len(costs)
    is_pareto = np.ones(n, dtype=bool)

    for i in range(n):
        if is_pareto[i]:
            dominated = (
                np.all(costs[is_pareto] <= costs[i], axis=1) &
                np.any(costs[is_pareto] <  costs[i], axis=1)
            )
            dominated_idx = np.where(is_pareto)[0][dominated]
            is_pareto[dominated_idx] = False
            is_pareto[i] = True

    return is_pareto

mask = pareto_front_fast(costs)
print(f"\nPareto front: {mask.sum()} materials out of {len(df_clean)}")

# ── 7. Weighted score ─────────────────────────────────────────────────────────

# Normalize full dataset to [0, 1]
scaler = MinMaxScaler()
df_normalized = pd.DataFrame(
    scaler.fit_transform(df_clean[cols]),
    columns=cols,
    index=df_clean.index
)

# Adjust weights to reflect your priorities — must sum to 1
weights = {
    "average_voltage":    0.10,
    "capacity_grav":      0.15,
    "energy_grav":        0.20,
    "fracA_charge":       0.05,
    "fracA_discharge":    0.05,
    "stability_charge":   0.08,
    "stability_discharge":0.07,
    "max_delta_volume":   0.05,
}

score = np.zeros(len(df_normalized))
for col, w in weights.items():
    if objectives[col] == "max":
        score += w * df_normalized[col]
    else:
        score += w * (1 - df_normalized[col])

# ── 8. Build ranked results table ─────────────────────────────────────────────

df_pareto = df_clean[mask].copy()
df_pareto["weighted_score"] = score[mask]

df_ranked = (
    df_pareto
    .sort_values("weighted_score", ascending=False)
    .reset_index(drop=True)
)
df_ranked.index += 1

# ── 9. Print table (TSV — paste directly into Google Sheets) ──────────────────

def print_ranked_materials(df_ranked, top_n=20):
    df_ranked["battery_id"] = df_ranked["battery_id"].astype(str)
    df_ranked["formula_discharge"] = df_ranked["formula_discharge"].astype(str)

    df_display = df_ranked.head(top_n)[[
        "weighted_score",
        "battery_id",
        "formula_discharge",
        "average_voltage",
        "capacity_grav",
        "energy_grav",
        "fracA_charge",
        "fracA_discharge",
        "stability_charge",
        "stability_discharge",
        "max_delta_volume",
    ]].round(3)

    df_display.columns = [
        "Score",
        "Battery ID",
        "Formula",
        "Voltage",
        "Cap_grav",
        "E_grav",
        "FracA_ch",
        "FracA_dis",
        "Stab_ch",
        "Stab_dis",
        "dVol%",
    ]

    print(f"\nTop {top_n} Pareto-optimal materials:\n")
    print(tabulate(
        df_display,
        headers="keys",
        tablefmt="tsv",
        floatfmt=".3f",
        numalign="right",
        stralign="left",
    ))

print_ranked_materials(df_ranked, top_n=20)

# ── 10. Save to CSV (for Google Sheets import) ────────────────────────────────

df_ranked.to_csv("pareto_results.csv", index=True, float_format="%.3f")
print("\nSaved to pareto_results.csv")


Retrieving InsertionElectrodeDoc documents: 100%|██████████| 2647/2647 [00:01<00:00, 2499.13it/s]

Materials after cleaning: 2647

Pareto front: 555 materials out of 2647

Top 20 Pareto-optimal materials:

  	  Score	Battery ID  	Formula         	  Voltage	  Cap_grav	  E_grav	  FracA_ch	  FracA_dis	  Stab_ch	  Stab_dis	  dVol%
 1	  0.537	mp-754868_Li	Li2FeO2         	    2.842	   526.936	1497.399	     0.000	      0.400	    0.257	     0.044	  0.307
 2	  0.519	mp-759420_Li	Li3MnF5         	    4.371	   313.921	1372.001	     0.143	      0.333	    0.023	     0.047	  0.188
 3	  0.497	mp-758209_Li	Li2FeSiO4       	    3.983	   331.271	1319.286	     0.000	      0.250	    0.262	     0.005	  0.087
 4	  0.489	mp-18968_Li 	Li2FeSiO4       	    4.049	   331.271	1341.162	     0.000	      0.250	    0.277	     0.000	  0.177
 5	  0.473	mp-26437_Li 	Li5Ni2(PO4)3    	    4.614	   245.319	1131.883	     0.056	      0.227	    0.140	     0.049	  0.057
 6	  0.473	mp-26275_Li 	Li5Ni2(PO4)3    	    4.686	   245.319	1149.457	     0.056	      0.227	    0.164	     0.056	  0.041
 7	  0.473	mp-9244_Li  	LiBC    